In [8]:
import requests
import pandas as pd

In [9]:
BACKEND_URL = "http://localhost:5000"

# API Functions
def check_backend_health():
    try:
        response = requests.get(f"{BACKEND_URL}/api/health", timeout=5)
        return response.status_code == 200
    except:
        return False

def get_tower_data():
    try:
        response = requests.get(f"{BACKEND_URL}/api/towers", timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['success']:
                return pd.DataFrame(data['data'])
        return pd.DataFrame()
    except Exception as e:
        st.error(f"Error fetching tower data: {str(e)}")
        return pd.DataFrame()

def get_predicted_outage_data():
    try:
        response = requests.get(f"{BACKEND_URL}/api/predicted-outages", timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['success']:
                return pd.DataFrame(data['data'])
        return pd.DataFrame()
    except Exception as e:
        st.error(f"Error fetching predicted outage data: {str(e)}")
        return pd.DataFrame()

def get_deployment_notes(cell_tower):
    try:
        response = requests.get(f"{BACKEND_URL}/api/deployment-notes", params={'tower_id': cell_tower['tower_id']}, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['success']:
                return data['data']
        return pd.DataFrame()
    except Exception as e:
        st.error(f"Error fetching deployment notes: {str(e)}")
        return pd.DataFrame()

In [10]:
import folium
import json

In [11]:
tower_data = get_tower_data()
tower_data.head()

,bandwidth,coverage_radius,latitude,longitude,signal_strength,status,technology,tower_id
0,80,3.4191801077287014,33.73912361050797,-93.85021150427312,-66.67120112977972,Active,5G,FN-3177
1,40,4.935219294382668,30.553976545593862,-104.50922091505767,-105.73073813756493,Active,5G,FN-5860
2,20,4.9576234363809615,29.67253609858686,-104.21580909487233,-109.32103264054298,Active,5G,FN-2961
3,80,2.550577782524861,28.65929051830793,-97.73941499270072,-117.05006846764701,Active,5G,FN-1183
4,20,4.878034978964565,33.045731382181344,-106.19200042088795,-76.32892042802756,Active,5G,FN-2640


In [12]:
outage_data = get_predicted_outage_data()
outage_data

,center_latitude,center_longitude,event,outage_id,radius,severity,towers_affected,towers_total
0,35.256223516367555,-103.96969679792443,Wildfire,Outage-5,41.214396087356086,Low,0,0
1,33.36064305276954,-105.93055104476602,Flood,Outage-7,17.974764676860985,Low,0,0
2,29.888784397520524,-101.89116241431171,Flood,Outage-1,42.470518822602685,Low,0,0
3,26.441773478876495,-96.3810145380411,Wildfire,Outage-4,42.45169106278405,Low,0,0
4,34.08597919911382,-93.88092568286626,Wildfire,Outage-2,36.51192959010121,High,0,0
5,29.207980242325814,-96.81809197319838,Flood,Outage-10,13.995869111379875,Low,0,0
6,33.239560485559636,-96.64485937889015,Wildfire,Outage-0,24.671836349254882,Medium,0,0
7,33.11139701990353,-100.43262695154856,Wildfire,Outage-6,22.43652705635785,Medium,0,0
8,31.816643991220648,-97.79241806912736,Wildfire,Outage-19,10.153853309118805,Critical,0,0
9,31.938651200806646,-102.26427034420402,Cyber Attack,Outage-12,40.411596987596226,Low,0,0


In [ ]:

from IPython.display import display

m = folium.Map(location=(31.96, -99.9), zoom_start=7, tiles="cartodb positron")

severity_2_color = {
    'Critical': 'darkred',
    'High': 'red',
    'Medium': 'orange',
    'Low': 'Yellow'
}

status_2_color = {
    'Active': 'green',
    'Down': 'red'
}

for _, row in tower_data.iterrows():
    if row['status'] != 'Down':
        continue
    
    popup_text = ""
    popup_text += f"<strong>Tower ID:</strong> {row['tower_id']}<br>"
    popup_text += f"<strong>Status:</strong> {row['status']}<br>"
    popup_text += f"<strong>Signal Strength:</strong> {float(row['signal_strength']):.3f} dBm<br>"
    popup_text += f"<strong>Bandwidth:</strong> {row['bandwidth']} MHz<br>"
    popup_text += f"<strong>Technology:</strong> {row['technology']}<br>"
    popup_text += f"<strong>Coverage:</strong> {row['coverage_radius']} mi<br>"
    popup_text += f"<strong>Lon:</strong> {float(row['longitude']):.4f}<br>"
    popup_text += f"<strong>Lat:</strong> {float(row['latitude']):.4f}<br>"
    
    popup = folium.Popup(popup_text, max_width=150)
    
    folium.Marker(
        location=[float(row['latitude']), float(row['longitude'])],
        tooltip=row['tower_id'],
        popup=popup,
        icon=folium.Icon(color=status_2_color[row['status']])
    ).add_to(m)

for _, row in outage_data.iterrows():
    popup_text = ""
    popup_text += f"<strong>Outage ID:</strong> {row['outage_id']}<br>"
    popup_text += f"<strong>Event:</strong> {row['event']}<br>"
    popup_text += f"<strong>Severity:</strong> {row['severity']}<br>"
    popup_text += f"<strong>Number of Total Towers:</strong> {row['towers_total']}<br>"
    popup_text += f"<strong>Number of Affected Towers:</strong> {row['towers_affected']}<br>"
    popup_text += f"<strong>Lon:</strong> {float(row['center_longitude']):.4f}<br>"
    popup_text += f"<strong>Lat:</strong> {float(row['center_latitude']):.4f}<br>"
    popup_text += f"<strong>Radius:</strong> {float(row['radius']):.2f} km<br>"
    
    popup = folium.Popup(popup_text, max_width=150)
    
    folium.Circle(
        location=[float(row['center_latitude']), float(row['center_longitude'])],
        radius=float(row['radius']) * 1000,  # Convert to meters
        color=severity_2_color[row['severity']],
        fill=True,
        fill_color=severity_2_color[row['severity']],
        tooltip=row['outage_id'],
        popup=popup
    ).add_to(m)

display(m)
